# Writing Data into MYSQL

In [1]:
import os
print("SPARK_HOME:", os.environ.get("SPARK_HOME"))
print("HADOOP_HOME:", os.environ.get("HADOOP_HOME"))

SPARK_HOME: C:\spark\spark-3.4.1-bin-hadoop3\spark-3.4.1-bin-hadoop3
HADOOP_HOME: C:\hadoop


In [2]:
import pandas as pd
salary = pd.read_csv("Salary_Data_Based_country_and_race.csv")
salary

,Unnamed: 0,Age,Gender,Education Level,Job Title,Years of Experience,Salary,Country,Race
0,0,32.0,Male,Bachelor's,Software Engineer,5.0,90000.0,UK,White
1,1,28.0,Female,Master's,Data Analyst,3.0,65000.0,USA,Hispanic
2,2,45.0,Male,PhD,Senior Manager,15.0,150000.0,Canada,White
3,3,36.0,Female,Bachelor's,Sales Associate,7.0,60000.0,USA,Hispanic
4,4,52.0,Male,Master's,Director,20.0,200000.0,USA,Asian
...,...,...,...,...,...,...,...,...,...
6699,6699,49.0,Female,PhD,Director of Marketing,20.0,200000.0,UK,Mixed
6700,6700,32.0,Male,High School,Sales Associate,3.0,50000.0,Australia,Australian
6701,6701,30.0,Female,Bachelor's Degree,Financial Manager,4.0,55000.0,China,Chinese
6702,6702,46.0,Male,Master's Degree,Marketing Manager,14.0,140000.0,China,Korean


In [3]:
import mysql.connector
#establis a connection with msql
db_connection = mysql.connector.connect(host="localhost", user="me", password="me")
db_cursor = db_connection.cursor()

# Create a new database
db_cursor.execute("CREATE DATABASE IF NOT EXISTS SPARK_DATA_2;")
db_cursor.execute("USE SPARK_DATA_2;")

# Create the table with corrected column names
create_table_query = """
CREATE TABLE IF NOT EXISTS salary_data_1 (
    Age FLOAT,
    GENDER VARCHAR(10),
    Education_Level VARCHAR(25),
    Job_Title VARCHAR(50),
    Years_of_Experience FLOAT,
    Salary FLOAT,
    Country VARCHAR(25),
    Race VARCHAR(25)
);
"""
db_cursor.execute(create_table_query) #the

In [11]:
import mysql.connector
import pandas as pd
import numpy as np

# Read the CSV file into a DataFrame
salary = pd.read_csv("Salary_Data_Based_country_and_race.csv")

# Drop rows with missing values (before handling missing values only we can give into mysql database ptherwise eror)
salary.dropna(subset=["Age", "Gender", "Education Level", "Job Title", "Years of Experience", "Salary", "Country", "Race"], inplace=True)

# Establish a connection to the database
db_connection = mysql.connector.connect(host="localhost", user="me", password="me")
db_cursor = db_connection.cursor()
db_cursor.execute("USE SPARK_DATA_2;")  # Making use of SPARK_DATA_2

# Convert DataFrame into a list of dictionaries
salary_records = salary.to_dict(orient="records")

# Define the INSERT query with properly escaped column names
insert_query = "INSERT INTO salary_data_1(Age, Gender, Education_Level, Job_Title, Years_of_Experience, Salary, Country, Race) VALUES (%s, %s, %s, %s, %s, %s, %s, %s)"

# Insert data into the database using parameterized query
for record in salary_records:
    values = (
        record['Age'], record['Gender'], record['Education Level'], record['Job Title'],
        record['Years of Experience'], record['Salary'], record['Country'], record['Race'])
    db_cursor.execute(insert_query, values)

# Commit the changes to the database
db_connection.commit()

# Close the cursor and database connection
db_cursor.close()
db_connection.close()


In [3]:
import findspark
findspark.init()

In [4]:
import mysql.connector
import pandas as pd
from pyspark.sql import SparkSession
#create a session with master as local 
spark = SparkSession.builder.master("local").appName("SalaryAnalysis").getOrCreate()
# Establish a connection
conn = mysql.connector.connect(user='me', database='SPARK_DATA_2',password='me',host="localhost",port=3306)
cursor = conn.cursor()

In [5]:
query="SELECT * FROM salary_data_1;"

In [6]:
pdf = pd.read_sql(query, con=conn)

C:\Users\Dayana Vincent\AppData\Local\Temp\ipykernel_5928\2975385224.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pdf = pd.read_sql(query, con=conn)


In [7]:
conn.close()

In [8]:
# Convert Pandas dataframe to spark DataFrame
df = spark.createDataFrame(pdf)

In [41]:
df.printSchema()

root
 |-- Age: double (nullable = true)
 |-- GENDER: string (nullable = true)
 |-- Education_Level: string (nullable = true)
 |-- Job_Title: string (nullable = true)
 |-- Years_of_Experience: double (nullable = true)
 |-- Salary: double (nullable = true)
 |-- Country: string (nullable = true)
 |-- Race: string (nullable = true)



In [10]:
df.show(6)

+----+------+---------------+-----------------+-------------------+--------+-------+--------+
| Age|GENDER|Education_Level|        Job_Title|Years_of_Experience|  Salary|Country|    Race|
+----+------+---------------+-----------------+-------------------+--------+-------+--------+
|32.0|  Male|     Bachelor's|Software Engineer|                5.0| 90000.0|     UK|   White|
|28.0|Female|       Master's|     Data Analyst|                3.0| 65000.0|    USA|Hispanic|
|45.0|  Male|            PhD|   Senior Manager|               15.0|150000.0| Canada|   White|
|36.0|Female|     Bachelor's|  Sales Associate|                7.0| 60000.0|    USA|Hispanic|
|52.0|  Male|       Master's|         Director|               20.0|200000.0|    USA|   Asian|
|29.0|  Male|     Bachelor's|Marketing Analyst|                2.0| 55000.0|    USA|Hispanic|
+----+------+---------------+-----------------+-------------------+--------+-------+--------+
only showing top 6 rows



# Statistical Analysis in Spark

In [9]:
df.describe().toPandas()

,summary,Age,GENDER,Education_Level,Job_Title,Years_of_Experience,Salary,Country,Race
0,count,6698,6698,6698,6698,6698,6698,6698,6698
1,mean,33.62302179755151,None,None,None,8.095177664974619,115329.25306061511,None,None
2,stddev,7.615784184052514,None,None,None,6.060291057757072,52789.7925069995,None,None
3,min,21.0,Female,Bachelor's,Account Manager,0.0,350.0,Australia,African American
4,max,62.0,Other,phD,Web Developer,34.0,250000.0,USA,White


# Data Cleaning

In [11]:
from pyspark.sql.functions import when,lit,count,isnan,col
df.select([count(when(isnan(c)|col(c).isNull(),c)).alias(c) for c in df.columns]).show()

+---+------+---------------+---------+-------------------+------+-------+----+
|Age|GENDER|Education_Level|Job_Title|Years_of_Experience|Salary|Country|Race|
+---+------+---------------+---------+-------------------+------+-------+----+
|  0|     0|              0|        0|                  0|     0|      0|   0|
+---+------+---------------+---------+-------------------+------+-------+----+



# Feature Vectors in Spark

In [28]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder
from pyspark.ml import Pipeline

string_cols = ['GENDER', 'Education_Level', 'Job_Title', 'Country', 'Race']
stages = []

# Loop through each string column and add stages for StringIndexer and OneHotEncoder
for col in string_cols:
    indexer = StringIndexer(inputCol=col, outputCol=f"{col}_index")
    encoder = OneHotEncoder(inputCol=f"{col}_index", outputCol=f"{col}_encoded")
    stages += [indexer, encoder]

# Assemble features
assembler = VectorAssembler(inputCols=[f"{col}_encoded" for col in string_cols],
                            outputCol="features")

stages += [assembler]

# Create a pipeline
pipeline = Pipeline(stages=stages)

# Fit and transform the data using the pipeline
transformed_data = pipeline.fit(df).transform(df)



# Assembling Model and Pipeline

In [29]:
from pyspark.ml.regression import LinearRegression
model = LinearRegression(featuresCol='features', labelCol='Salary')

In [30]:
# Split the data into training and testing sets
train_data, test_data = transformed_data.randomSplit([0.7, 0.3], seed=42)

In [31]:
# Fit the model to the training data
modell = model.fit(train_data)

In [32]:
# Make predictions on the test data
predictions = modell.transform(test_data)

In [51]:
display_data = predictions.select(features + ["Salary", 'prediction'])
display_data.show(10)

+----+------+---------------+--------------------+-------------------+---------+----------------+-------+------------------+
| Age|GENDER|Education_Level|           Job_Title|Years_of_Experience|  Country|            Race| Salary|        prediction|
+----+------+---------------+--------------------+-------------------+---------+----------------+-------+------------------+
|21.0|Female|    High School|Junior Sales Repr...|                0.0|Australia|      Australian|25000.0| 23378.01551455143|
|21.0|Female|    High School|Junior Sales Repr...|                0.0|   Canada|           Black|25000.0|21104.177258233074|
|21.0|Female|    High School|Junior Sales Repr...|                0.0|    China|         Chinese|25000.0|20194.406138187536|
|21.0|Female|    High School|Junior Sales Repr...|                0.0|    China|          Korean|25000.0|19834.439833448036|
|21.0|Female|    High School|Junior Sales Repr...|                0.0|       UK|           White|25000.0| 25436.93561310004|


In [50]:
data_scientist_1=predictions.filter(predictions.Job_Title=="Data Scientist")
s= data_scientist_1.select(features + ['Job_Title',"Salary", 'prediction'])
s.show(10)

+----+------+---------------+--------------+-------------------+---------+------+--------------+--------+------------------+
| Age|GENDER|Education_Level|     Job_Title|Years_of_Experience|  Country|  Race|     Job_Title|  Salary|        prediction|
+----+------+---------------+--------------+-------------------+---------+------+--------------+--------+------------------+
|27.0|  Male|            PhD|Data Scientist|                2.0|Australia| White|Data Scientist|115000.0|169702.23327162946|
|27.0|  Male|            PhD|Data Scientist|                2.0|       UK| Mixed|Data Scientist|115000.0|167911.97021406627|
|28.0|Female|       Master's|Data Scientist|                6.0|   Canada| White|Data Scientist|150000.0|179525.01459066285|
|28.0|Female|       Master's|Data Scientist|                6.0|       UK| Asian|Data Scientist|150000.0|177721.97582375287|
|28.0|Female|       Master's|Data Scientist|                6.0|      USA| Asian|Data Scientist|150000.0|177295.48109117206|


In [49]:
lr_predictions = modell.transform(test_data)
lr_predictions.select("prediction","salary","features").show(10)

+------------------+-------+--------------------+
|        prediction| salary|            features|
+------------------+-------+--------------------+
| 23378.01551455143|25000.0|(211,[1,6,51,200,...|
|21104.177258233074|25000.0|(211,[1,6,51,207]...|
|20194.406138187536|25000.0|(211,[1,6,51,199,...|
|19834.439833448036|25000.0|(211,[1,6,51,199,...|
| 25436.93561310004|25000.0|(211,[1,6,51,201,...|
|19356.462457334506|25000.0|(211,[1,6,51,198,...|
|19356.462457334506|25000.0|(211,[1,6,51,198,...|
| 73723.90832223365|51832.0|(211,[1,6,16,198,...|
| 26199.62548171141|25000.0|(211,[1,6,41,200,...|
| 24775.13663546933|25000.0|(211,[1,6,41,203]...|
+------------------+-------+--------------------+
only showing top 10 rows



# Evaluation Metrics

In [42]:
trainingSummary = modell.summary
print("RMSE: %f" % trainingSummary.rootMeanSquaredError)
print("r2: %f" % trainingSummary.r2)

RMSE: 27024.341443
r2: 0.741529


R squared at 0.74 indicates that in our model, approximate 74% of the variability in “salary” can be explained using the model.. It is not bad

The performance on the training set may not a good approximation of the performance on the test set then do the same for test data.

In [53]:
print("numIterations: %d" % trainingSummary.totalIterations)
print("objectiveHistory: %s" % str(trainingSummary.objectiveHistory))
trainingSummary.residuals.show(10)

numIterations: 85
objectiveHistory: [0.5, 0.3347924006709455, 0.14251246186068522, 0.13679116297479954, 0.13207423389426864, 0.13071073688200396, 0.12978281593016217, 0.12961472530382512, 0.12954753941854547, 0.1295290533200677, 0.1295078827100209, 0.129503357597335, 0.12949498986842922, 0.1294864312802364, 0.1294747268856571, 0.1294570906797743, 0.1294460720008992, 0.12943160476280768, 0.12941926388699398, 0.1294031335138972, 0.12938494692916835, 0.12936859402220557, 0.12936033221411325, 0.12934898142983586, 0.12933476412315725, 0.12929941317077898, 0.12928650798766883, 0.12925972318965373, 0.1292485188267749, 0.1292476314127966, 0.129246539817772, 0.12924636989719884, 0.12924609500199002, 0.12924600782756412, 0.12924598173147794, 0.12924595534376726, 0.12924593989058053, 0.12924590225986154, 0.12924584269009776, 0.12924582001686424, 0.12924570239685051, 0.1292456398376669, 0.12924555618799705, 0.12924543825099644, 0.1292453582706634, 0.12924521374954523, 0.12924513400196513, 0.129245